# Data Dictionary — Rename & Drop Columns

This notebook takes raw data dictionary CSVs downloaded from the data freeze portal and transforms them into the format required by the submission guide. It handles all files at once so you don't have to process each one manually.

---

### What it does
1. Loads all data dictionary CSVs from `input/dd_downloads/` (or a single file if you prefer)
2. Strips each file down to only the columns the submission guide asks for
3. Renames columns to match the submission schema and inserts a blank `units` column in the right position
4. Exports clean, consistently named files to `output/data_dictionary/`

---

### Required folder structure

```
datafreeze_ETL_scripts/
├── input/
│   ├── dd_downloads/           <- place data dictionary CSV files here
│   └── fw_downloads/           <- UDS data CSV files (used by data_freeze_filter.ipynb)
├── output/
│   ├── data_dictionary/        <- output from this notebook (auto-created)
│   └── uds_data/               <- output from data_freeze_filter.ipynb
└── notebooks/
    └── data_dictionary.ipynb   <- you are here
```

> **Note:** Downloaded filenames include a date stamp that changes with every new freeze — the notebook picks them up automatically and strips the date from the exported filename so your output names stay consistent.

---

### Two ways to run

| Mode | When to use |
|---|---|
| `"folder"` | Process every CSV in `dd_downloads/` at once — the most common use case |
| `"standalone"` | Process a single file — handy for a quick check or one-off submission |

Start by filling in your run mode in the **Configuration** cell below.

In [1]:
# Standard libraries used throughout the notebook.
# re handles the filename cleanup (stripping date codes).
# pathlib makes file path handling cleaner across operating systems.
import re
from pathlib import Path

import pandas as pd

## Configuration

Fill in the cell below before running anything else.

- **Run mode** — choose whether to process all files at once or just one
- **Paths** — these should work as-is if your folder structure matches the diagram above

In [2]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# "folder"     -> picks up every CSV in dd_downloads/ automatically
# "standalone" -> processes only the file named in SINGLE_FILE below
MODE        = "folder"
SINGLE_FILE = "uds-v4-ivp-ded-04142026.csv"  # only used when MODE = "standalone"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Leave these as-is if your folder structure matches the diagram in the intro.
# Only change them if you've moved files around.
DOWNLOADS_DIR = Path("../input/dd_downloads")
OUTPUT_DIR    = Path("../output/data_dictionary")

## Load the files

Scans the downloads folder and loads every CSV it finds into memory. Each file gets its own dataframe so they can be processed and exported independently.

In [3]:
# Folder mode picks up every CSV in dd_downloads/ without you needing to
# know the exact filenames — useful since downloaded files have a date stamp
# appended that changes with each new data freeze.
# Standalone mode loads just the one file you named above.
if MODE == "standalone":
    path = DOWNLOADS_DIR / SINGLE_FILE
    dataframes = {path.name: pd.read_csv(path)}
else:
    dataframes = {
        f.name: pd.read_csv(f)
        for f in sorted(DOWNLOADS_DIR.glob("*.csv"))
    }

# A quick row and column count per file — good to glance at before transforming
# to make sure everything loaded as expected.
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

bds-v1-ded-10142025.csv: 52 rows, 8 columns
cls-v3-ded-10142025.csv: 19 rows, 8 columns
ftld-v3-fvp-ded-04142026.csv: 367 rows, 10 columns
ftld-v3-ivp-ded-04142026.csv: 367 rows, 10 columns
lbd-v3.0-fvp-ded-02252025.csv: 464 rows, 10 columns
lbd-v3.0-ivp-ded-02252025.csv: 463 rows, 10 columns
lbd-v3.1-fvp-ded-02252025.csv: 301 rows, 10 columns
lbd-v3.1-ivp-ded-02252025.csv: 300 rows, 10 columns
milestones-v3-ded-10142025 (1).csv: 33 rows, 8 columns
np-v11-ded-10142025 (1).csv: 164 rows, 8 columns
sample.csv: 3 rows, 10 columns
uds-v4-fvp-ded-04142026.csv: 1461 rows, 12 columns
uds-v4-ivp-ded-04142026.csv: 1535 rows, 10 columns


## Strip to core columns

Drops everything that isn't needed for the submission. Also handles a known inconsistency where a small number of forms use `form_name` instead of `packet` — those get normalized here so the rest of the pipeline treats all files the same.

In [4]:
# The full list of columns the submission requires. Everything else in the
# source files — branching logic, missingness flags, notes, etc. — gets dropped.
CORE_COLUMNS = ["packet", "var_name", "question", "conformity", "response_labels", "data_type"]

def strip_to_core(df):
    # A small number of forms don't have a "packet" column and use "form_name"
    # instead. We rename it here so all files share the same column structure
    # going forward. The check at the bottom tells you which files this applied to.
    if "packet" not in df.columns:
        df = df.rename(columns={"form_name": "packet"})
        return df[CORE_COLUMNS], True
    return df[CORE_COLUMNS], False

defaulted = []
result = {}
for name, df in dataframes.items():
    result[name], used_form_name = strip_to_core(df)
    if used_form_name:
        defaulted.append(name)

dataframes = result

# Worth a quick look — if an unexpected file shows up here it may mean
# the source format changed between data freeze versions.
print("Used form_name instead of packet:", defaulted if defaulted else "none")

# Preview the first file to confirm the columns look right before moving on.
next(iter(dataframes.values())).head()

Used form_name instead of packet: ['bds-v1-ded-10142025.csv', 'cls-v3-ded-10142025.csv', 'milestones-v3-ded-10142025 (1).csv', 'np-v11-ded-10142025 (1).csv']


,packet,var_name,question,conformity,response_labels,data_type
0,bds,adcid,0a. ADRC ID,List of current ADCIDs,List of current ADCIDs,Integer
1,bds,ptid,0b. PTID,String with max length of 10 characters,NaN,String
2,bds,visitdate,0c. Form Date,mm/dd/yyyy or yyyy/mm/dd,NaN,Date
3,bds,initials,0d. Examiner's initials,Any text,NaN,String
4,bds,formver,0e. Form version number,1,"1, 1",Numeric


## Rename columns and add Units

Translates the source column names to the ones the submission guide expects, and inserts a blank `units` column in the correct position. The source files don't have a units column at all, so it's added here as an empty field for you to fill in where applicable.

In [5]:
# Maps source column names to the names the submission guide expects.
# "units" is not a rename — it's a new blank column inserted at the right position.
RENAME_MAP = {
    "var_name":   "variable_name",
    "question":   "description",
    "conformity": "key_description",
}

def rename_columns(df):
    df = df.rename(columns=RENAME_MAP)
    # Inserted just before "key_description" to match the column order in the guide.
    df.insert(df.columns.get_loc("key_description"), "units", "")
    return df

dataframes = {name: rename_columns(df) for name, df in dataframes.items()}

# Preview to confirm the final column order looks right before exporting.
next(iter(dataframes.values())).head()

,packet,variable_name,description,units,key_description,response_labels,data_type
0,bds,adcid,0a. ADRC ID,,List of current ADCIDs,List of current ADCIDs,Integer
1,bds,ptid,0b. PTID,,String with max length of 10 characters,NaN,String
2,bds,visitdate,0c. Form Date,,mm/dd/yyyy or yyyy/mm/dd,NaN,Date
3,bds,initials,0d. Examiner's initials,,Any text,NaN,String
4,bds,formver,0e. Form version number,,1,"1, 1",Numeric


## Export the transformed files

Saves each transformed dataframe as a CSV in `output/data_dictionary/`. The date stamp baked into the downloaded filename is stripped out so the exported names stay short and consistent across freeze versions.

In [6]:
# Creates the output folder if it isn't there yet — no need to create it manually.
OUTPUT_DIR.mkdir(exist_ok=True)

def clean_filename(name):
    # Downloaded files come with a date stamp in the name
    # (e.g. "uds-v4-ivp-ded-04142026.csv").
    # We strip that out here so the saved file has a clean, stable name
    # that won't be different the next time a new freeze is downloaded.
    stem = Path(name).stem
    stem = re.sub(r"\s*\(\d+\)", "", stem)  # remove " (1)" from accidental duplicate downloads
    stem = re.sub(r"-ded-\d+", "", stem)        # remove "-ded-MMDDYYYY" style date suffixes
    return stem + ".csv"

for name, df in dataframes.items():
    out_name = clean_filename(name)
    df.to_csv(OUTPUT_DIR / out_name, index=False)
    print(f"{name}  ->  {out_name}")

bds-v1-ded-10142025.csv  ->  bds-v1.csv
cls-v3-ded-10142025.csv  ->  cls-v3.csv
ftld-v3-fvp-ded-04142026.csv  ->  ftld-v3-fvp.csv
ftld-v3-ivp-ded-04142026.csv  ->  ftld-v3-ivp.csv
lbd-v3.0-fvp-ded-02252025.csv  ->  lbd-v3.0-fvp.csv
lbd-v3.0-ivp-ded-02252025.csv  ->  lbd-v3.0-ivp.csv
lbd-v3.1-fvp-ded-02252025.csv  ->  lbd-v3.1-fvp.csv
lbd-v3.1-ivp-ded-02252025.csv  ->  lbd-v3.1-ivp.csv
milestones-v3-ded-10142025 (1).csv  ->  milestones-v3.csv
np-v11-ded-10142025 (1).csv  ->  np-v11.csv
sample.csv  ->  sample.csv
uds-v4-fvp-ded-04142026.csv  ->  uds-v4-fvp.csv
uds-v4-ivp-ded-04142026.csv  ->  uds-v4-ivp.csv
